In [ ]:
# Import libraries
import glob
import rasterio
from rasterio.warp import calculate_default_transform, reproject
import numpy as np
import pandas as pd
from affine import Affine

In [ ]:
import os
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd
from rasterstats import zonal_stats

In [ ]:
import re

In [ ]:
# 1. Load farm boundaries (Albers Equal Area)
gdf = gpd.read_file('GeoFiles/CSB/CSB_AZ.shp')
# Fix geometries (apply buffer(0) to geometry column)
gdf.geometry = gdf.geometry.buffer(0)  # Fix invalid polygons
print("Farm CRS:", gdf.crs)  # Should be EPSG:5070

In [ ]:
# 2. Configuration
target_crs = "EPSG:5070"
resolution = (30, 30)  # Tuple for reprojection
results = []

In [ ]:
# 3. Validate
print("Farm CRS:", gdf.crs)
print("Columns:", gdf.columns.tolist())

# Check CDL columns dynamically
cdl_columns = [col for col in gdf.columns if re.match(r'CDL\d{4}', col)]
missing_cdl = [f"CDL{year}" for year in range(2017, 2025) if f"CDL{year}" not in cdl_columns]
print(f"Missing CDL Columns: {missing_cdl if missing_cdl else 'None'}")

# Geometry check
print(f"Geometry Valid: {gdf.geometry.is_valid.all()}")

In [ ]:
# 3. Process each NDVI raster
for raster_path in glob.glob('./30m/Arizona_NDVI_*.tif'):
    with rasterio.open(raster_path) as src:
        print(f"\nProcessing {os.path.basename(raster_path)}")

        # Get source metadata
        src_crs = src.crs.to_string()
        src_nodata = src.nodata if src.nodata is not None else -9999

        # Reproject if needed
        if src_crs != target_crs:
            # Calculate transform with forced 30m resolution
            transform, width, height = calculate_default_transform(
                src.crs, target_crs,
                src.width, src.height,
                *src.bounds,
                resolution=resolution
            )

            # Create destination array
            raster_data = np.empty((height, width), dtype=np.float32)

            # Perform reprojection
            reproject(
                source=src.read(1),
                destination=raster_data,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=target_crs,
                resampling=rasterio.enums.Resampling.bilinear,
                src_nodata=src_nodata,
                dst_nodata=np.nan
            )
            affine = transform  # Directly use the calculated transform
        else:
            raster_data = src.read(1)
            affine = src.transform

        # Parse metadata from filename
        parts = os.path.basename(raster_path).split('_')
        stat_mode = parts[2].lower()
        year = int(parts[4])
        month = int(parts[5].split('.')[0])

        # Calculate zonal statistics
        stats = zonal_stats(
            gdf,
            raster_data,
            affine=affine,  # Use the transform directly
            stats=['mean'],
            nodata=np.nan,
            geojson_out=True
        )

        # Store results
        for feat in stats:
            props = feat['properties']
            # Find the matching row in gdf (assuming 'CSBID' is unique)
            original_props = gdf[gdf['CSBID'] == props['CSBID']].iloc[0].to_dict()
            # Make a copy of all shapefile attributes
            feature_dict = original_props.copy()
            # Overwrite/add a 'CDL' property based on the current year (e.g., 'CDL2020')
            cdl_year_col = f'CDL{year}'
            if cdl_year_col in feature_dict:
              feature_dict['CDL'] = feature_dict[cdl_year_col]
            else:
              feature_dict['CDL'] = None  # or a placeholder if year is missing
            # Add the NDVI value (mean, median, min, etc. depending on your code)
            feature_dict['NDVI'] = props['mean']

            # Add time/context columns
            feature_dict['year'] = year
            feature_dict['month'] = month
            feature_dict['stat_type'] = stat_mode

            # Append to the final results list
            results.append(feature_dict)

In [ ]:
df = pd.DataFrame(results)

In [ ]:
df

In [ ]:
df['CSBID'] = df['CSBID'].astype(str)

In [ ]:
for column in df.columns:
    print(column)

In [ ]:
df[['CSBID','CNTY','Shape_Area','CDL','NDVI','year','month','stat_type']]

In [ ]:
df.dtypes

In [ ]:
df.to_csv('Data/AT_ET_farm_data.csv', index=False)

In [ ]:
df2 = pd.read_csv('Data/AT_ET_farm_data.csv')

In [ ]:
df2

In [ ]:
# Check if the combination of CSBID, year, and month is unique
duplicates_exist = df2.duplicated(subset=['CSBID', 'year', 'month']).any()
are_unique = not duplicates_exist

if are_unique:
    print("All combinations of CSBID, year, and month are unique!")
else:
    print("WARNING: Duplicates exist in the combinations of CSBID, year, and month.")

In [ ]:
df2.dtypes

In [ ]:
# Check unique lengths in the reloaded DataFrame
# Convert the int64 column to string (to check lengths)
df2['CSBID_str'] = df2['CSBID'].astype(str)#.str.zfill(15)
reloaded_lengths = df2['CSBID_str'].str.len().unique()
print("Reloaded CSBID lengths:", reloaded_lengths)  # Might show [14, 15] or other values